In [5]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

In [3]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

# ----------------------------
# 1️⃣ Create a sample DataFrame with missing values
# ----------------------------
data_dict = {
    'Feature1': [1.2, np.nan, 3.4, 4.5, 5.6],
    'Feature2': [2.3, 3.5, np.nan, 5.6, 6.1],
    'Feature3': [np.nan, 1.0, 2.2, 3.3, 4.4],
    'Feature4': [7.8, 8.9, 9.0, np.nan, 10.2],
}
df = pd.DataFrame(data_dict)

# ----------------------------
# 2️⃣ Initial mean imputation — this acts like Iteration 0
# ----------------------------
initial_data = df.copy()
for col in initial_data.columns:
    initial_data[col] = initial_data[col].fillna(initial_data[col].mean())

iteration_0 = initial_data.copy()  # Save for comparison

# ----------------------------
# 3️⃣ Manual regression-based update for one missing value (simulate iteration 1)
# ----------------------------
# Reintroduce the missing value in Feature1
initial_data.loc[df['Feature1'].isnull(), 'Feature1'] = np.nan

# Train on rows where Feature1 is known
x_train = initial_data.loc[[0, 2, 3, 4], ['Feature2', 'Feature3', 'Feature4']]
y_train = initial_data.loc[[0, 2, 3, 4], 'Feature1']

# Predict the missing Feature1 value (row 1)
x_test = initial_data.loc[1, ['Feature2', 'Feature3', 'Feature4']].values.reshape(1, -1)

lr = LinearRegression()
lr.fit(x_train, y_train)
predicted_value = lr.predict(x_test)[0]

# Fill the predicted value back
initial_data.loc[1, 'Feature1'] = predicted_value

# repeate this for as many items as diff is near 0


# using sklearn
# ----------------------------
# 4️⃣ Run MICE (IterativeImputer) for 3 different iteration counts
# ----------------------------
results = []
max_iters = [50, 100, 150]  # simulate multiple MICE runs

for iters in max_iters:
    imputer = IterativeImputer(estimator=LinearRegression(), max_iter=iters, random_state=42)
    imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    imputed_df.columns = [f"{col}_iter{iters}" for col in imputed_df.columns]
    results.append(imputed_df)

# ----------------------------
# 5️⃣ Combine all imputed datasets (iteration_0, iter50, iter100, iter150)
# ----------------------------
iteration_0.columns = [f"{col}_iter0" for col in iteration_0.columns]
combined = pd.concat([iteration_0] + results, axis=1)

# View the combined result
print(combined)
